In [1]:
import nfl_data_py as nfl


In [2]:
years = [2024]
pbp_df = nfl.import_pbp_data(years, downcast=True, cache=False, alt_path=None)


2024 done.
Downcasting floats.


In [3]:
time_related_columns = ['time_of_day']
pbp_df[time_related_columns]

,time_of_day
0,None
1,2024-09-08T17:03:02.957Z
2,2024-09-08T17:03:40.463Z
3,2024-09-08T17:04:12.743Z
4,2024-09-08T17:04:57.067Z
...,...
49487,2025-02-10T03:14:44.040Z
49488,2025-02-10T03:15:28.947Z
49489,2025-02-10T03:16:09.927Z
49490,2025-02-10T03:16:48.130Z


In [4]:
for col in pbp_df.columns:
    print(col)

play_id
game_id
old_game_id_x
home_team
away_team
season_type
week
posteam
posteam_type
defteam
side_of_field
yardline_100
game_date
quarter_seconds_remaining
half_seconds_remaining
game_seconds_remaining
game_half
quarter_end
drive
sp
qtr
down
goal_to_go
time
yrdln
ydstogo
ydsnet
desc
play_type
yards_gained
shotgun
no_huddle
qb_dropback
qb_kneel
qb_spike
qb_scramble
pass_length
pass_location
air_yards
yards_after_catch
run_location
run_gap
field_goal_result
kick_distance
extra_point_result
two_point_conv_result
home_timeouts_remaining
away_timeouts_remaining
timeout
timeout_team
td_team
td_player_name
td_player_id
posteam_timeouts_remaining
defteam_timeouts_remaining
total_home_score
total_away_score
posteam_score
defteam_score
score_differential
posteam_score_post
defteam_score_post
score_differential_post
no_score_prob
opp_fg_prob
opp_safety_prob
opp_td_prob
fg_prob
safety_prob
td_prob
extra_point_prob
two_point_conversion_prob
ep
epa
total_home_epa
total_away_epa
total_home_rush_ep

In [61]:
from supabase import create_client, Client
import os
import nfl_data_py as nfl
import pandas as pd

def get_nfl_player_id_to_espn_id_map(years):
    try:
        roster_data = nfl.import_seasonal_rosters(years)
        player_id_to_espn_id_map = {}
        for index, row in roster_data.iterrows():
            if pd.notna(row['player_id']) and pd.notna(row['espn_id']):
                player_id_to_espn_id_map[row['player_id']] = row['espn_id']
        return player_id_to_espn_id_map
    except Exception as e:
        return {}

def get_supabase_client():
    try:
        # load supabase client    
        supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
        return supabase
    except Exception as e:
        raise

def get_espn_player_id_to_player_id_map(supabase_client):
    try:
        response = supabase_client.table("nfl_players").select("id, espn_player_id").execute()
        player_map = {}
        for player in response.data:
            if player.get("espn_player_id") is not None and player.get("id") is not None:
                player_map[player["espn_player_id"]] = player["id"]
        return player_map
    except Exception as e:
        return {}

def get_big_play_schema():
    return {
        'player_id': '',
        'timestamp': '',
        'description': '',
    }

def safe_get_espn_id(nfl_player_id, nfl_to_espn_map, player_type=""):
    """Safely get ESPN ID from NFL player ID with error handling"""
    if not nfl_player_id or pd.isna(nfl_player_id):
        return None
    
    if nfl_player_id not in nfl_to_espn_map:
        return None
    
    return nfl_to_espn_map[nfl_player_id]

def safe_get_db_id(espn_player_id, espn_to_db_map, player_type=""):
    """Safely get DB ID from ESPN player ID with error handling"""
    if not espn_player_id:
        return None
    
    if espn_player_id not in espn_to_db_map:
        return None
    
    return espn_to_db_map[espn_player_id]

def create_big_play(db_player_id, timestamp, description):
    """Create a big play entry with proper validation"""
    if not db_player_id or not timestamp:
        return None
    
    big_play = get_big_play_schema()
    big_play['player_id'] = db_player_id
    big_play['timestamp'] = timestamp
    big_play['description'] = description
    return big_play

def get_all_big_play_timestamps(week, year):
    try:
        years = [year]
        
        # Import play-by-play data
        pbp_df = nfl.import_pbp_data(years, downcast=True, cache=False, alt_path=None)
        
        # Get client and mappings
        supabase_client = get_supabase_client()
        espn_player_id_to_player_id_map = get_espn_player_id_to_player_id_map(supabase_client)
        nfl_player_id_to_espn_id_map = get_nfl_player_id_to_espn_id_map(years)
        
        print("Espn player id to player id map: " + str(espn_player_id_to_player_id_map))
        
        # Filter for specific week
        week_pbp_df = pbp_df[pbp_df['week'] == week]
        
        big_plays = []
        skipped_plays = 0
        
        for index, row in week_pbp_df.iterrows():
            try:
                if row['pass_attempt'] == 1:
                    # Check for passing big plays (TD or 30+ yards)
                    if row.get('pass_touchdown') == 1 or (pd.notna(row.get('yards_gained')) and row['yards_gained'] > 30):
                        
                        # Process passer
                        passer_espn_id = safe_get_espn_id(row.get('passer_player_id'), nfl_player_id_to_espn_id_map, "passer")
                        passer_db_id = safe_get_db_id(int(passer_espn_id), espn_player_id_to_player_id_map, "passer")
                        
                        # Process receiver
                        receiver_espn_id = safe_get_espn_id(row.get('receiver_player_id'), nfl_player_id_to_espn_id_map, "receiver")
                        receiver_db_id = safe_get_db_id(int(receiver_espn_id), espn_player_id_to_player_id_map, "receiver")
                        
                        yards_gained = row.get('yards_gained', 0)
                        is_touchdown = row.get('pass_touchdown') == 1
                        
                        # Create passer big play if valid
                        if passer_db_id:
                            description = f"Passing {'touchdown' if is_touchdown else 'completion'} ({int(yards_gained)} yards)"
                            qb_big_play = create_big_play(passer_db_id, row.get('time_of_day'), description)
                            if qb_big_play:
                                big_plays.append(qb_big_play)
                        else:
                            skipped_plays += 1
                        
                        # Create receiver big play if valid
                        if receiver_db_id:
                            description = f"Receiving {'touchdown' if is_touchdown else 'completion'} ({int(yards_gained)} yards)"
                            wr_big_play = create_big_play(receiver_db_id, row.get('time_of_day'), description)
                            if wr_big_play:
                                big_plays.append(wr_big_play)
                        else:
                            skipped_plays += 1
                
                elif row['rush_attempt'] == 1:
                    # Check for rushing big plays (TD or 20+ yards)
                    if row.get('rush_touchdown') == 1 or (pd.notna(row.get('yards_gained')) and row['yards_gained'] > 20):
                        
                        # Process rusher
                        rusher_espn_id = safe_get_espn_id(row.get('rusher_player_id'), nfl_player_id_to_espn_id_map, "rusher")
                        rusher_db_id = safe_get_db_id(int(rusher_espn_id), espn_player_id_to_player_id_map, "rusher")
                        
                        if rusher_db_id:
                            yards_gained = row.get('yards_gained', 0)
                            is_touchdown = row.get('rush_touchdown') == 1
                            description = f"Rushing {'touchdown' if is_touchdown else 'play'} ({int(yards_gained)} yards)"
                            
                            running_big_play = create_big_play(rusher_db_id, row.get('time_of_day'), description)
                            if running_big_play:
                                big_plays.append(running_big_play)
                        else:
                            skipped_plays += 1
            
            except Exception as e:
                skipped_plays += 1
                continue
        
        return big_plays
    
    except Exception as e:
        return []

def insert_big_plays(big_plays):
    try:
        supabase_client = get_supabase_client()
        supabase_client.table("nfl_big_plays").insert(big_plays).execute()
        return True
    except Exception as e:
        print(e)
        return False

In [62]:
week = 1
year = 2024

all_big_plays = get_all_big_play_timestamps(week, year)

2024 done.
Downcasting floats.
Espn player id to player id map: {4426535: '878c783e-9d6d-483c-927f-166612ada5fa', 4241389: 'a6441ae5-9b15-40e1-99a5-0daf4657bb01', 4374302: '0e27646d-0f43-4dec-86ca-31cfd6270b05', 3916387: 'e60bad2f-cdf9-4f3b-80d3-d117bb9b244d', 3918298: 'b4d1f53f-515f-440e-9475-1335873e385f', 3043078: 'bf8800cd-902e-4680-afff-76196d607c3b', 4426515: '7f6cc29f-8e36-47fc-89bd-977b1c591526', 4595348: 'b8d5eebb-bb78-41b1-9364-8382cd976c4d', 4047646: 'b876aeff-ceda-40d1-8b28-b17f450a2e83', 4040715: 'aa9f0898-8520-402f-a57e-e1dc2e0d1ccb', 4361307: '170fb224-4bc9-4580-b93b-6533e11b5c7e', 3040151: 'adf848ce-a025-4a4b-a562-610d06c4665b', 4258173: 'f0ea896b-666c-4c7f-a1ec-6ca5213b0df1', 4047365: '2a5ed905-c0c6-44dc-b0ed-2c2ab96e1fc1', 4242335: 'fe6e82e3-8e79-49c1-a85a-23b964796fbf', 3915511: 'ed73b236-c4f0-458e-aad2-3dc83a299110', 4239993: '99ad3924-5b1d-41d0-9350-252e4e30010e', 4426502: 'e79f56e7-8fdc-4395-bc3b-995d063f80d1', 4890973: 'c87b9168-091f-4e25-be74-fdc479df7dc9', 4430

In [63]:
insert_big_plays(all_big_plays)

True

In [56]:
all_big_plays

[{'espn_player_id': '3917315',
  'db_player_id': 'a4e6deba-e8ec-4b58-ad35-63a777acb08f',
  'timestamp': '2024-09-08T17:15:46.637Z',
  'description': 'Passing touchdown (5 yards)'},
 {'espn_player_id': '4360761',
  'db_player_id': '5f710670-39de-48a1-a0ba-3cfd355c5149',
  'timestamp': '2024-09-08T17:15:46.637Z',
  'description': 'Receiving touchdown (5 yards)'},
 {'espn_player_id': '3045147',
  'db_player_id': '1daa9168-af2d-44cd-9566-7e2bf2779803',
  'timestamp': '2024-09-08T17:57:28.880Z',
  'description': 'Rushing touchdown (3 yards)'},
 {'espn_player_id': '3918298',
  'db_player_id': 'b4d1f53f-515f-440e-9475-1335873e385f',
  'timestamp': '2024-09-08T18:16:19.833Z',
  'description': 'Rushing touchdown (7 yards)'},
 {'espn_player_id': '3918298',
  'db_player_id': 'b4d1f53f-515f-440e-9475-1335873e385f',
  'timestamp': '2024-09-08T18:41:43.390Z',
  'description': 'Passing touchdown (11 yards)'},
 {'espn_player_id': '2991662',
  'db_player_id': '42f0a18d-bddb-4355-8395-b83d771db1d8',
  '